# Scoring des detections de navires

Ce notebook applique le scoring retenu pour prioriser les detections. Le score est volontairement simple : il sert a trier les observations, pas a mesurer une menace reelle.

## Regle de scoring

Le score final est borne a 100 points. Certaines lignes sont exclusives entre elles : une zone ne peut pas etre a la fois `Critical` et `High`, et une detection ne prend qu'un seul bonus de confiance.

### Maximum theorique : 100 points

- Navire militaire : +35.
- Niveau de zone, maximum +25 :
  - +25 si la zone est `Critical` ;
  - +15 si la zone est `High`.
- Confiance, maximum +15 :
  - +15 si la confiance est superieure ou egale a 0.85 ;
  - +8 si la confiance est entre 0.75 et 0.85.
- Type strategique : +15 si le type est porte-avions, sous-marin, destroyer ou croiseur.
- Proximite : +10 si la zone militaire la plus proche est active et a moins de 25 km.

Somme maximale possible : 35 + 25 + 15 + 15 + 10 = 100.

Le fichier final exporte est `detections_enrichies_scoring.csv`.


In [25]:
from pathlib import Path
import math

import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

In [26]:
GENERALISATION_DIR = Path.cwd()

# Si le notebook est lance depuis la racine Sujet5 au lieu du dossier Generalisation,
# on retrouve le dossier par la presence des fichiers source.
if not (GENERALISATION_DIR / 'detection_results.csv').exists():
    GENERALISATION_DIR = next(
        p for p in Path.cwd().iterdir()
        if p.is_dir() and (p / 'detection_results.csv').exists() and (p / 'military_zones.csv').exists()
    )

DETECTIONS_PATH = GENERALISATION_DIR / 'detection_results.csv'
IMAGES_PATH = GENERALISATION_DIR / 'images_metadata_large.csv'
ZONES_PATH = GENERALISATION_DIR / 'military_zones.csv'
OUTPUT_PATH = GENERALISATION_DIR / 'detections_enrichies_scoring.csv'

print(GENERALISATION_DIR.resolve())

C:\Users\louni\Desktop\hackaton marine\Hackaton_Marine_2026\Sujet5\Généralisation


In [27]:
detections = pd.read_csv(DETECTIONS_PATH)
images = pd.read_csv(IMAGES_PATH)
zones = pd.read_csv(ZONES_PATH)

print(f'Detections : {len(detections)}')
print(f'Images : {len(images)}')
print(f'Zones militaires : {len(zones)}')

detections.head()

Detections : 256
Images : 100
Zones militaires : 20


,detection_id,image_id,file_name,timestamp,bbox,category,confidence,is_military,zone_id,zone_name,risk_level
0,DET-000-00001,IMG-000,satellite_000.jpg,2026-08-31T17:17:58Z,"0.3766847153545144,0.602830810224785,0.2798675...",Navire de guerre,0.84,True,ZONE-002,Base navale de Norfolk,High
1,DET-000-00002,IMG-000,satellite_000.jpg,2026-08-31T17:17:58Z,"0.6730405409693083,0.37685587994514613,0.29411...",Navire civil,0.72,False,ZONE-002,Base navale de Norfolk,High
2,DET-000-00003,IMG-000,satellite_000.jpg,2026-08-31T17:17:58Z,"0.3499607760017571,0.6063398574692226,0.244706...",Bâtiment de débarquement,0.96,False,ZONE-002,Base navale de Norfolk,High
3,DET-000-00004,IMG-000,satellite_000.jpg,2026-08-31T17:17:58Z,"0.6159782694606443,0.45954657746946426,0.27517...",Destroyer,0.89,True,ZONE-002,Base navale de Norfolk,High
4,DET-000-00005,IMG-000,satellite_000.jpg,2026-08-31T17:17:58Z,"0.7200515440830189,0.2160546184846373,0.135625...",Pétrolier,0.90,False,ZONE-002,Base navale de Norfolk,High


In [28]:
def parse_coordinates(value):
    lat, lon = str(value).split(',')
    return float(lat), float(lon)

def parse_bbox(value):
    x, y, w, h = [float(v) for v in str(value).split(',')]
    return pd.Series({
        'bbox_x': x,
        'bbox_y': y,
        'bbox_width': w,
        'bbox_height': h,
        'bbox_area': w * h,
        'bbox_aspect_ratio': w / h if h else pd.NA,
    })

def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * radius_km * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [29]:
image_coords = images['coordinates'].apply(parse_coordinates)
images[['image_latitude', 'image_longitude']] = pd.DataFrame(image_coords.tolist(), index=images.index)

zone_coords = zones['coordinates'].apply(parse_coordinates)
zones[['zone_latitude', 'zone_longitude']] = pd.DataFrame(zone_coords.tolist(), index=zones.index)
zones['active'] = zones['active'].astype(bool)

image_columns = [
    'image_id', 'date', 'time', 'coordinates', 'country', 'resolution', 'source',
    'num_ships', 'ship_types', 'cloud_cover', 'url', 'image_latitude', 'image_longitude'
]

scored = detections.merge(images[image_columns], on='image_id', how='left')
scored = pd.concat([scored, scored['bbox'].apply(parse_bbox)], axis=1)

scored[['detection_id', 'image_id', 'category', 'confidence', 'zone_name', 'risk_level', 'image_latitude', 'image_longitude']].head()

,detection_id,image_id,category,confidence,zone_name,risk_level,image_latitude,image_longitude
0,DET-000-00001,IMG-000,Navire de guerre,0.84,Base navale de Norfolk,High,36.9,-76.3
1,DET-000-00002,IMG-000,Navire civil,0.72,Base navale de Norfolk,High,36.9,-76.3
2,DET-000-00003,IMG-000,Bâtiment de débarquement,0.96,Base navale de Norfolk,High,36.9,-76.3
3,DET-000-00004,IMG-000,Destroyer,0.89,Base navale de Norfolk,High,36.9,-76.3
4,DET-000-00005,IMG-000,Pétrolier,0.90,Base navale de Norfolk,High,36.9,-76.3


In [30]:
active_zones = zones[zones['active']].copy()

nearest_active_rows = []
for row in scored.itertuples(index=False):
    distances = active_zones.apply(
        lambda zone: haversine_km(
            row.image_latitude,
            row.image_longitude,
            zone['zone_latitude'],
            zone['zone_longitude'],
        ),
        axis=1,
    )
    nearest_idx = distances.idxmin()
    nearest_zone = active_zones.loc[nearest_idx]
    nearest_active_rows.append({
        'nearest_active_military_zone_id': nearest_zone['zone_id'],
        'nearest_active_military_zone_name': nearest_zone['name'],
        'nearest_active_military_zone_risk_level': nearest_zone['risk_level'],
        'nearest_active_military_zone_distance_km': round(float(distances.loc[nearest_idx]), 3),
    })

scored = pd.concat([scored, pd.DataFrame(nearest_active_rows, index=scored.index)], axis=1)
scored['near_active_military_zone_25km'] = scored['nearest_active_military_zone_distance_km'] <= 25

scored[['detection_id', 'zone_name', 'nearest_active_military_zone_name', 'nearest_active_military_zone_distance_km', 'near_active_military_zone_25km']].head()

,detection_id,zone_name,nearest_active_military_zone_name,nearest_active_military_zone_distance_km,near_active_military_zone_25km
0,DET-000-00001,Base navale de Norfolk,Zone militaire 11 - Port militaire de Toulon,6747.776,False
1,DET-000-00002,Base navale de Norfolk,Zone militaire 11 - Port militaire de Toulon,6747.776,False
2,DET-000-00003,Base navale de Norfolk,Zone militaire 11 - Port militaire de Toulon,6747.776,False
3,DET-000-00004,Base navale de Norfolk,Zone militaire 11 - Port militaire de Toulon,6747.776,False
4,DET-000-00005,Base navale de Norfolk,Zone militaire 11 - Port militaire de Toulon,6747.776,False


In [31]:
strategic_types = {'Porte-avions', 'Sous-marin', 'Destroyer', 'Croiseur'}

scored['score_military'] = scored['is_military'].astype(bool).astype(int) * 35
scored['score_risk_level'] = scored['risk_level'].map({'Critical': 25, 'High': 15}).fillna(0).astype(int)
scored['score_confidence'] = scored['confidence'].apply(lambda confidence: 15 if confidence >= 0.85 else 8 if confidence >= 0.75 else 0)
scored['score_strategic_type'] = scored['category'].isin(strategic_types).astype(int) * 15
scored['score_active_zone_25km'] = scored['near_active_military_zone_25km'].astype(int) * 10

score_columns = [
    'score_military',
    'score_risk_level',
    'score_confidence',
    'score_strategic_type',
    'score_active_zone_25km',
]
scored['priority_score'] = scored[score_columns].sum(axis=1)

scored['priority_label'] = pd.cut(
    scored['priority_score'],
    bins=[-1, 39, 69, 84, 100],
    labels=['Faible', 'Moyenne', 'Haute', 'Critique'],
)

scored[score_columns + ['priority_score', 'priority_label']].head()

,score_military,score_risk_level,score_confidence,score_strategic_type,score_active_zone_25km,priority_score,priority_label
0,35,15,8,0,0,58,Moyenne
1,0,15,0,0,0,15,Faible
2,0,15,15,0,0,30,Faible
3,35,15,15,15,0,80,Haute
4,0,15,15,0,0,30,Faible


In [32]:
front_columns = [
    'priority_score',
    'priority_label',
    'detection_id',
    'image_id',
    'file_name',
    'timestamp',
    'category',
    'confidence',
    'is_military',
    'zone_id',
    'zone_name',
    'risk_level',
    'image_latitude',
    'image_longitude',
    'nearest_active_military_zone_id',
    'nearest_active_military_zone_name',
    'nearest_active_military_zone_risk_level',
    'nearest_active_military_zone_distance_km',
    'near_active_military_zone_25km',
]
remaining_columns = [column for column in scored.columns if column not in front_columns]

scored = scored[front_columns + remaining_columns].sort_values(
    ['priority_score', 'confidence'],
    ascending=[False, False],
).reset_index(drop=True)

scored.sample(10)

,priority_score,priority_label,detection_id,image_id,file_name,timestamp,category,confidence,is_military,zone_id,zone_name,risk_level,image_latitude,image_longitude,nearest_active_military_zone_id,nearest_active_military_zone_name,nearest_active_military_zone_risk_level,nearest_active_military_zone_distance_km,near_active_military_zone_25km,bbox,date,time,coordinates,country,resolution,source,num_ships,ship_types,cloud_cover,url,bbox_x,bbox_y,bbox_width,bbox_height,bbox_area,bbox_aspect_ratio,score_military,score_risk_level,score_confidence,score_strategic_type,score_active_zone_25km
75,65,Moyenne,DET-004-00012,IMG-004,satellite_004.jpg,2026-10-03T08:53:49Z,Destroyer,0.96,True,ZONE-015,Mer Baltique,Medium,55.0,20.0,MIL-010,Zone militaire 11 - Port militaire de Toulon,High,1666.902,False,"0.7459333595512729,0.10487580225891656,0.05163...",2026-10-03,08:53:49,"55.0,20.0",International,1m,Sentinel-2,3,"Destroyer, Pétrolier, Frégate",32.7,https://example.com/satellite_images/satellite...,0.745933,0.104876,0.051638,0.193957,0.010016,0.266235,35,0,15,15,0
242,8,Faible,DET-070-00193,IMG-070,satellite_070.jpg,2026-10-08T12:35:10Z,Navire de soutien,0.80,False,ZONE-006,Port de Rotterdam,Low,51.9,4.5,MIL-010,Zone militaire 11 - Port militaire de Toulon,High,981.887,False,"0.723673278123738,0.7160937693222476,0.2118303...",2026-10-08,12:35:10,"51.9,4.5",Netherlands,10m,Landsat-8,5,"Navire de guerre, Destroyer, Chalutier, Navire...",6.2,https://example.com/satellite_images/satellite...,0.723673,0.716094,0.211830,0.155548,0.032950,1.361835,0,0,8,0,0
108,53,Moyenne,DET-037-00117,IMG-037,satellite_037.jpg,2026-05-03T20:00:00Z,Corvette,0.79,True,ZONE-003,Port de Shanghai,Medium,31.2,121.5,MIL-001,Zone militaire 2 - Port de Shanghai,Critical,0.000,True,"0.47804213404009466,0.1398971181387041,0.11920...",2026-05-03,20:00:00,"31.2,121.5",China,10m,Landsat-8,5,"Navire de soutien, Corvette, Navire de guerre,...",7.0,https://example.com/satellite_images/satellite...,0.478042,0.139897,0.119205,0.082659,0.009853,1.442129,35,0,8,0,10
45,75,Haute,DET-033-00105,IMG-033,satellite_033.jpg,2026-01-13T19:16:01Z,Navire de guerre,0.95,True,ZONE-004,Détroit de Malacca,Critical,3.0,101.0,MIL-001,Zone militaire 2 - Port de Shanghai,Critical,3801.407,False,"0.41423056710094325,0.18503203450789352,0.1665...",2026-01-13,19:16:01,"3.0,101.0",International,5m,Maxar,5,"Destroyer, Frégate, Navire de guerre, Corvette...",39.2,https://example.com/satellite_images/satellite...,0.414231,0.185032,0.166535,0.229391,0.038202,0.725989,35,25,15,0,0
118,50,Moyenne,DET-070-00190,IMG-070,satellite_070.jpg,2026-10-08T12:35:10Z,Navire de guerre,0.91,True,ZONE-006,Port de Rotterdam,Low,51.9,4.5,MIL-010,Zone militaire 11 - Port militaire de Toulon,High,981.887,False,"0.3370364814651254,0.4338078266533149,0.025100...",2026-10-08,12:35:10,"51.9,4.5",Netherlands,10m,Landsat-8,5,"Navire de guerre, Destroyer, Chalutier, Navire...",6.2,https://example.com/satellite_images/satellite...,0.337036,0.433808,0.025100,0.179912,0.004516,0.139515,35,0,15,0,0
128,45,Moyenne,DET-007-00024,IMG-007,satellite_007.jpg,2026-04-06T03:08:14Z,Navire de guerre,0.72,True,ZONE-003,Port de Shanghai,Medium,31.2,121.5,MIL-001,Zone militaire 2 - Port de Shanghai,Critical,0.000,True,"0.016630245559289138,0.6261170766235215,0.1213...",2026-04-06,03:08:14,"31.2,121.5",China,5m,WorldView-3,5,"Croiseur, Navire de guerre, Chalutier, Pétroli...",21.1,https://example.com/satellite_images/satellite...,0.016630,0.626117,0.121329,0.025770,0.003127,4.708185,35,0,0,0,10
184,25,Faible,DET-007-00025,IMG-007,satellite_007.jpg,2026-04-06T03:08:14Z,Chalutier,0.95,False,ZONE-003,Port de Shanghai,Medium,31.2,121.5,MIL-001,Zone militaire 2 - Port de Shanghai,Critical,0.000,True,"0.3285007396795342,0.5734258448954177,0.083891...",2026-04-06,03:08:14,"31.2,121.5",China,5m,WorldView-3,5,"Croiseur, Navire de guerre, Chalutier, Pétroli...",21.1,https://example.com/satellite_images/satellite...,0.328501,0.573426,0.083892,0.251807,0.021124,0.33

In [33]:
summary = pd.DataFrame({
    'nb_detections': [len(scored)],
    'score_min': [scored['priority_score'].min()],
    'score_max': [scored['priority_score'].max()],
    'score_moyen': [round(scored['priority_score'].mean(), 2)],
    'nb_score_critique': [(scored['priority_label'] == 'Critique').sum()],
    'nb_zone_active_25km': [scored['near_active_military_zone_25km'].sum()],
})
summary

,nb_detections,score_min,score_max,score_moyen,nb_score_critique,nb_zone_active_25km
0,256,0,100,47.44,21,44


In [34]:
# Sanity check : on verifie que le score exporte est coherent.
expected_score = scored[score_columns].sum(axis=1)
score_mismatch = scored.loc[scored['priority_score'] != expected_score, ['detection_id', 'priority_score'] + score_columns]

assert score_mismatch.empty, 'Des lignes ont un priority_score different de la somme des composantes.'
assert len(scored) == len(detections), 'Le nombre de lignes scorees doit rester identique aux detections source.'
assert scored['priority_score'].between(0, 100).all(), 'Le priority_score doit rester entre 0 et 100.'
assert scored['detection_id'].is_unique, 'Chaque detection_id doit rester unique.'
assert scored['priority_label'].notna().all(), 'Chaque ligne doit avoir un priority_label.'

allowed_values = {
    'score_military': {0, 35},
    'score_risk_level': {0, 15, 25},
    'score_confidence': {0, 8, 15},
    'score_strategic_type': {0, 15},
    'score_active_zone_25km': {0, 10},
}
for column, allowed in allowed_values.items():
    unexpected = set(scored[column].dropna().unique()) - allowed
    assert not unexpected, f'Valeurs inattendues dans {column}: {unexpected}'

near_zone_check = scored['near_active_military_zone_25km'].eq(
    scored['nearest_active_military_zone_distance_km'] <= 25
)
assert near_zone_check.all(), 'La colonne near_active_military_zone_25km ne correspond pas a la distance <= 25 km.'

constant_components = [column for column in score_columns if scored[column].nunique(dropna=False) == 1]
print('Sanity check OK')
print(f'Lignes controlees : {len(scored)}')
print(f'Score min / max : {scored["priority_score"].min()} / {scored["priority_score"].max()}')
print('Maximum theorique : 35 + 25 + 15 + 15 + 10 = 100')
if constant_components:
    print('Composantes constantes dans ce jeu de donnees : ' + ', '.join(constant_components))
else:
    print('Aucune composante de score constante.')


Sanity check OK
Lignes controlees : 256
Score min / max : 0 / 100
Maximum theorique : 35 + 25 + 15 + 15 + 10 = 100
Aucune composante de score constante.


In [35]:
scored.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f'Export termine : {OUTPUT_PATH.resolve()}')

Export termine : C:\Users\louni\Desktop\hackaton marine\Hackaton_Marine_2026\Sujet5\Généralisation\detections_enrichies_scoring.csv
